# Self-Supervised Learning: SimCLR-style Contrastive Pretraining

This notebook implements a self-supervised learning experiment for the BostonGene IML-2 painting-style classification task.

The model is trained without using class labels. For each image, two augmented views are created. The encoder is trained to make the representations of two views of the same image similar, while making representations of different images less similar.

This experiment is added as a bonus self-supervised learning component. It does not replace the required supervised classifier or the required VAE embedding/clustering part.

In [ ]:
from pathlib import Path
import sys
import os
import shutil
import zipfile

# ============================================================
# 1. Detect environment
# ============================================================

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

    DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/bostongene_project")
    DRIVE_ZIP = DRIVE_PROJECT_DIR / "bostongene_classifier_bundle.zip"

    print("Drive project folder:", DRIVE_PROJECT_DIR)
    print("Expected zip:", DRIVE_ZIP)

    assert DRIVE_PROJECT_DIR.exists(), f"Drive folder not found: {DRIVE_PROJECT_DIR}"
    assert DRIVE_ZIP.exists(), f"Zip file not found: {DRIVE_ZIP}"

    WORK_DIR = Path("/content/bostongene_project")

    if WORK_DIR.exists():
        shutil.rmtree(WORK_DIR)

    WORK_DIR.mkdir(parents=True, exist_ok=True)

    print("Extracting project zip...")

    with zipfile.ZipFile(DRIVE_ZIP, "r") as zip_ref:
        for member in zip_ref.namelist():
            correct_path = member.replace("\\", "/")
            target_path = WORK_DIR / correct_path

            if correct_path.endswith("/"):
                target_path.mkdir(parents=True, exist_ok=True)
            else:
                target_path.parent.mkdir(parents=True, exist_ok=True)
                with zip_ref.open(member) as source, open(target_path, "wb") as target:
                    target.write(source.read())

    print("Unzipped into:", WORK_DIR)

else:
    WORK_DIR = Path.cwd().resolve()

    if WORK_DIR.name == "notebooks":
        WORK_DIR = WORK_DIR.parent


# ============================================================
# 2. Find project root
# ============================================================

src_dirs = list(WORK_DIR.rglob("src"))

assert len(src_dirs) > 0, (
    "Could not find src directory. "
    "Make sure you launched the notebook from the project root or extracted the zip correctly."
)

PROJECT_ROOT = src_dirs[0].parent

assert (PROJECT_ROOT / "data").exists(), "data/ folder is missing."
assert (PROJECT_ROOT / "notebooks").exists(), "notebooks/ folder is missing."
assert (PROJECT_ROOT / "src" / "data.py").exists(), "src/data.py is missing."
assert (PROJECT_ROOT / "src" / "config.py").exists(), "src/config.py is missing."

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("IN_COLAB:", IN_COLAB)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("Current working directory:", Path.cwd())

In [ ]:
import random
import time
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import resnet18

from src.data import PadToSquare
from src.config import TRAIN_DIR, VAL_DIR, TEST_DIR, MODELS_DIR, OUTPUTS_DIR, FIGURES_DIR, NUM_CLASSES, SEED


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
print("TRAIN_DIR:", TRAIN_DIR)
print("VAL_DIR:", VAL_DIR)
print("TEST_DIR:", TEST_DIR)

assert TRAIN_DIR.exists(), f"Train directory not found: {TRAIN_DIR}"
assert VAL_DIR.exists(), f"Validation directory not found: {VAL_DIR}"
assert TEST_DIR.exists(), f"Test directory not found: {TEST_DIR}"

In [ ]:
IMG_SIZE_SSL = 128

# SimCLR benefits from large batches, but Colab GPU memory is limited.
# If you get CUDA out-of-memory, reduce this to 32.
SSL_BATCH_SIZE = 64 if device.type == "cuda" else 16

NUM_WORKERS_SSL = 2 if device.type == "cuda" else 0


class ContrastiveTransform:
    """
    Given one PIL image, return two independently augmented views.

    These two views form a positive pair for SimCLR.
    """

    def __init__(self, base_transform):
        self.base_transform = base_transform

    def __call__(self, image):
        view_1 = self.base_transform(image)
        view_2 = self.base_transform(image)
        return view_1, view_2


ssl_base_transform = transforms.Compose([
    PadToSquare(fill=(0, 0, 0)),
    transforms.RandomResizedCrop(
        size=IMG_SIZE_SSL,
        scale=(0.55, 1.0),
        ratio=(0.75, 1.33),
    ),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomApply(
        [
            transforms.ColorJitter(
                brightness=0.35,
                contrast=0.35,
                saturation=0.35,
                hue=0.08,
            )
        ],
        p=0.8,
    ),
    transforms.RandomGrayscale(p=0.2),
    transforms.GaussianBlur(kernel_size=9, sigma=(0.1, 2.0)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

ssl_dataset = datasets.ImageFolder(
    root=TRAIN_DIR,
    transform=ContrastiveTransform(ssl_base_transform),
)

ssl_loader = DataLoader(
    ssl_dataset,
    batch_size=SSL_BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS_SSL,
    pin_memory=(device.type == "cuda"),
    drop_last=True,
)

print("SSL training images:", len(ssl_dataset))
print("SSL batch size:", SSL_BATCH_SIZE)
print("Number of SSL batches:", len(ssl_loader))
print("Classes exist in dataset, but labels will be ignored during SSL:")
print(ssl_dataset.classes)

In [ ]:
(views_1, views_2), labels = next(iter(ssl_loader))

print("View 1 batch:", views_1.shape)
print("View 2 batch:", views_2.shape)
print("Labels shape:", labels.shape)
print("Labels are loaded but ignored during SimCLR pretraining.")

In [ ]:
class SimCLRModel(nn.Module):
    """
    ResNet18 encoder + projection head for SimCLR.

    The encoder learns image representations.
    The projection head is used only during contrastive pretraining.
    """

    def __init__(self, projection_dim: int = 128):
        super().__init__()

        backbone = resnet18(weights=None)

        feature_dim = backbone.fc.in_features
        backbone.fc = nn.Identity()

        self.encoder = backbone

        self.projection_head = nn.Sequential(
            nn.Linear(feature_dim, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, projection_dim),
        )

    def forward(self, x):
        h = self.encoder(x)
        z = self.projection_head(h)
        return h, z


simclr_model = SimCLRModel(projection_dim=128).to(device)

print(simclr_model)

In [ ]:
def nt_xent_loss(z1, z2, temperature: float = 0.5):
    """
    Normalized Temperature-scaled Cross Entropy Loss.

    z1 and z2 are projected representations of two augmented views
    of the same batch of images.
    """
    batch_size = z1.size(0)

    z = torch.cat([z1, z2], dim=0)
    z = F.normalize(z, dim=1)

    similarity_matrix = torch.matmul(z, z.T) / temperature

    # Remove self-similarity from the denominator
    mask = torch.eye(2 * batch_size, dtype=torch.bool, device=z.device)
    similarity_matrix = similarity_matrix.masked_fill(mask, -1e9)

    # Positive pair index:
    # first half matches second half, second half matches first half
    targets = torch.arange(2 * batch_size, device=z.device)
    targets = (targets + batch_size) % (2 * batch_size)

    loss = F.cross_entropy(similarity_matrix, targets)

    return loss

In [ ]:
EPOCHS_SSL = 1 if device.type == "cpu" else 50

optimizer_ssl = torch.optim.AdamW(
    simclr_model.parameters(),
    lr=3e-4,
    weight_decay=1e-4,
)

scheduler_ssl = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_ssl,
    T_max=EPOCHS_SSL,
)

best_ssl_loss = float("inf")
history_ssl = []

ssl_checkpoint_path = MODELS_DIR / "simclr_resnet18_ssl.pt"

print(f"Training SimCLR for {EPOCHS_SSL} epoch(s)")
print(f"Checkpoint will be saved to: {ssl_checkpoint_path}")

start_time = time.time()

for epoch in range(1, EPOCHS_SSL + 1):
    simclr_model.train()

    running_loss = 0.0
    n_samples = 0

    epoch_start = time.time()

    for (x1, x2), _ in ssl_loader:
        x1 = x1.to(device)
        x2 = x2.to(device)

        optimizer_ssl.zero_grad(set_to_none=True)

        _, z1 = simclr_model(x1)
        _, z2 = simclr_model(x2)

        loss = nt_xent_loss(z1, z2, temperature=0.5)

        loss.backward()
        optimizer_ssl.step()

        batch_size = x1.size(0)
        running_loss += loss.item() * batch_size
        n_samples += batch_size

    scheduler_ssl.step()

    epoch_loss = running_loss / n_samples
    current_lr = optimizer_ssl.param_groups[0]["lr"]

    history_ssl.append({
        "epoch": epoch,
        "ssl_loss": epoch_loss,
        "lr": current_lr,
    })

    improved = epoch_loss < best_ssl_loss

    if improved:
        best_ssl_loss = epoch_loss

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": simclr_model.state_dict(),
                "encoder_state_dict": simclr_model.encoder.state_dict(),
                "projection_head_state_dict": simclr_model.projection_head.state_dict(),
                "ssl_loss": best_ssl_loss,
                "config": {
                    "architecture": "ResNet18 SimCLR",
                    "img_size": IMG_SIZE_SSL,
                    "batch_size": SSL_BATCH_SIZE,
                    "projection_dim": 128,
                    "temperature": 0.5,
                    "pretraining_data": "train split only",
                    "labels_used": False,
                },
            },
            ssl_checkpoint_path,
        )

    epoch_time = time.time() - epoch_start

    print(
        f"Epoch {epoch:03d}/{EPOCHS_SSL} | "
        f"ssl_loss={epoch_loss:.4f} | "
        f"best_ssl_loss={best_ssl_loss:.4f} | "
        f"lr={current_lr:.6f} | "
        f"time={epoch_time:.1f}s"
    )

total_time = time.time() - start_time

print(f"SSL pretraining finished in {total_time / 60:.2f} minutes.")
print(f"Best SSL loss: {best_ssl_loss:.4f}")

In [ ]:
history_ssl_df = pd.DataFrame(history_ssl)

ssl_history_path = OUTPUTS_DIR / "simclr_ssl_pretraining_history.csv"
history_ssl_df.to_csv(ssl_history_path, index=False)

display(history_ssl_df.head())
display(history_ssl_df.tail())

print(f"Saved SSL history to: {ssl_history_path}")

fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(
    history_ssl_df["epoch"],
    history_ssl_df["ssl_loss"],
    marker="o",
    label="SimCLR SSL loss",
)

ax.set_title("SimCLR Self-Supervised Pretraining Loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("NT-Xent Loss")
ax.grid(True, alpha=0.3)
ax.legend()

fig.tight_layout()

ssl_loss_fig_path = FIGURES_DIR / "13_simclr_ssl_loss_curve.png"
fig.savefig(ssl_loss_fig_path, dpi=150, bbox_inches="tight")

plt.show()

print(f"Saved SSL loss curve to: {ssl_loss_fig_path}")

In [ ]:
from pathlib import Path
import shutil
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    DRIVE_RESULTS_DIR = Path("/content/drive/MyDrive/bostongene_project/simclr_results")
    DRIVE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

    for folder_name in ["models", "outputs", "reports"]:
        src = PROJECT_ROOT / folder_name
        dst = DRIVE_RESULTS_DIR / folder_name

        if src.exists():
            shutil.copytree(src, dst, dirs_exist_ok=True)
            print(f"Copied {src} -> {dst}")

    print("Saved SimCLR results to Google Drive:", DRIVE_RESULTS_DIR)
else:
    print("Not running in Colab, no Drive backup needed.")

In [ ]:
from pathlib import Path
import shutil
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    DRIVE_RESULTS_DIR = Path("/content/drive/MyDrive/bostongene_project/simclr_results")
    DRIVE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

    for folder_name in ["models", "outputs", "reports"]:
        src = PROJECT_ROOT / folder_name
        dst = DRIVE_RESULTS_DIR / folder_name

        if src.exists():
            shutil.copytree(src, dst, dirs_exist_ok=True)
            print(f"Copied {src} -> {dst}")

    print("Saved SimCLR results to Google Drive:", DRIVE_RESULTS_DIR)
else:
    print("Not running in Colab, no Drive backup needed.")